# Conversion Propensity Model: Personalization Scenario Lab

**IBM Z Datathon 2026 — hypothetical real-time personalization demo**

This notebook sends synthetic customer/session profiles to the Conversion Propensity API, compares the model's class probabilities, and maps each result to a **hypothetical** personalization action.

> **Important demo assumptions**
> - All profiles and actions in this notebook are synthetic and illustrative.
> - The propensity bands below are demonstration thresholds, not validated business thresholds. They should be calibrated using historical outcomes, campaign costs, and controlled experiments.
> - A prediction is not proof that an offer will cause conversion.

## Business question

How might a digital experience adapt its next message or offer to a visitor's predicted conversion propensity, while keeping the experience relevant and non-intrusive?

## 1. Connect to the API

Because this notebook runs on the same VM as FastAPI we use `127.0.0.1` rather than the VM's public IP.

In [1]:
import requests
import pandas as pd
from IPython.display import display, Markdown

BASE_URL = "http://127.0.0.1:8000"
PREDICT_URL = f"{BASE_URL}/predict"

health_response = requests.get(f"{BASE_URL}/health", timeout=10)
health_response.raise_for_status()
health = health_response.json()
print("API health:", health)

model_response = requests.get(f"{BASE_URL}/model-info", timeout=10)
model_response.raise_for_status()
model_info = model_response.json()
print("Model type:", model_info.get("model_type"))
print("Expected features:", model_info.get("features"))
print("Model classes:", model_info.get("classes"))

API health: {'status': 'ok', 'model_loaded': True}
Model type: Pipeline
Expected features: ['country', 'city', 'device_category', 'browser', 'first_user_default_channel_group', 'sessions', 'engaged_sessions', 'bounce_rate', 'average_session_duration', 'views', 'views_per_session', 'events_per_session', 'item_view_events', 'add_to_carts', 'checkouts']
Model classes: [0, 1]


## 2. Define synthetic customer scenarios

The profiles represent different observed session behaviors. They are **scenario hypotheses**, not claims about how the model must score them. The model's actual output determines the propensity band and proposed action.

In [2]:
scenarios = [
    {
        "scenario": "Quick Visit",
        "business_hypothesis": "A visitor arrives and leaves with little exploration.",
        "country": "United States", "city": "Chicago", "device_category": "desktop",
        "browser": "Chrome", "first_user_default_channel_group": "Organic Search",
        "sessions": 2, "engaged_sessions": 1, "bounce_rate": 0.5,
        "average_session_duration": 60.0, "views": 10, "views_per_session": 5.0,
        "events_per_session": 10.0, "item_view_events": 9, "add_to_carts": 0, "checkouts": 0,
    },
    {
        "scenario": "Product Explorer",
        "business_hypothesis": "A visitor browses several pages, views multiple products and adds them to the cart without reaching checkout page",
        "country": "United States", "city": "Chicago", "device_category": "desktop",
        "browser": "Chrome", "first_user_default_channel_group": "Organic Search",
        "sessions": 2, "engaged_sessions": 2, "bounce_rate": 0.0,
        "average_session_duration": 80.0, "views": 10, "views_per_session": 5.0,
        "events_per_session": 10.0, "item_view_events": 9, "add_to_carts": 9, "checkouts": 0,
    },
    {
        "scenario": "Engaged Shopper",
        "business_hypothesis": "A visitor is engaged and views multiple products and reaches the checkout page, suggesting active consideration.",
        "country": "United States", "city": "Chicago", "device_category": "desktop",
        "browser": "Chrome", "first_user_default_channel_group": "Organic Search",
        "sessions": 1, "engaged_sessions": 1, "bounce_rate": 0.0,
        "average_session_duration": 90.0, "views": 3, "views_per_session": 3.0,
        "events_per_session": 5.0, "item_view_events": 1, "add_to_carts": 5, "checkouts": 1,
    },
    {
        "scenario": "Active Checkout",
        "business_hypothesis": "A visitor has added items to the cart, started checkout and clearly demostrated purchase intentions",
        "country": "United States", "city": "Chicago", "device_category": "desktop",
        "browser": "Chrome", "first_user_default_channel_group": "Organic Search",
        "sessions": 2, "engaged_sessions": 2, "bounce_rate": 0.0,
        "average_session_duration": 120.0, "views": 10, "views_per_session": 5.0,
        "events_per_session": 10.0, "item_view_events": 9, "add_to_carts": 5, "checkouts": 2,
    }
]

FEATURES = model_info["features"]
scenario_preview = pd.DataFrame([
    {"Scenario": s["scenario"], **{f: s[f] for f in FEATURES}}
    for s in scenarios
])
display(scenario_preview)

,Scenario,country,city,device_category,browser,first_user_default_channel_group,sessions,engaged_sessions,bounce_rate,average_session_duration,views,views_per_session,events_per_session,item_view_events,add_to_carts,checkouts
0,Quick Visit,United States,Chicago,desktop,Chrome,Organic Search,2,1,0.5,60.0,10,5.0,10.0,9,0,0
1,Product Explorer,United States,Chicago,desktop,Chrome,Organic Search,2,2,0.0,80.0,10,5.0,10.0,9,9,0
2,Engaged Shopper,United States,Chicago,desktop,Chrome,Organic Search,1,1,0.0,90.0,3,3.0,5.0,1,5,1
3,Active Checkout,United States,Chicago,desktop,Chrome,Organic Search,2,2,0.0,120.0,10,5.0,10.0,9,5,2


## 3. Call the model for every scenario

For each profile, the notebook records the predicted class and both class probabilities. The API's `conversion_probability` is interpreted as the probability for class `1`, subject to the label-encoding assumption stated above.

In [3]:
results = []

for scenario in scenarios:
    features = {feature: scenario[feature] for feature in FEATURES}
    response = requests.post(
        PREDICT_URL,
        json={"features": features},
        timeout=60
    )
    response.raise_for_status()
    prediction = response.json()

    p0 = prediction["class_probabilities"].get("0")
    p1 = prediction["class_probabilities"].get("1")
    results.append({
        "Scenario": scenario["scenario"],
        "Predicted class": prediction["predicted_class"],
        "Class 0 probability": p0,
        "Class 1 probability": p1,
        "Conversion probability (%)": prediction.get("conversion_probability_percent")
    })

results_df = pd.DataFrame(results)
display(results_df.style.format({
    "Class 0 probability": "{:.1%}",
    "Class 1 probability": "{:.1%}",
    "Conversion probability (%)": "{:.2f}%",
}, na_rep="N/A"))

,Scenario,Predicted class,Class 0 probability,Class 1 probability,Conversion probability (%)
0,Quick Visit,0,97.0%,3.0%,3.00%
1,Product Explorer,0,76.0%,24.0%,24.00%
2,Engaged Shopper,1,48.0%,52.0%,52.00%
3,Active Checkout,1,22.0%,78.0%,78.00%


## 4. Convert propensity into a hypothetical decision

For this demo only, we use these illustrative bands:

| Class-1 probability | Demo band | Offer Strategy |
|---:|---|---|
| `>= 65%` | High | No discount by default. Make checkout easy and reinforce trust; this visitor may already be ready to convert. |
| `30%–<65%` | Medium | Test free shipping as the first incentive. If free shipping is unavailable or unsuitable, consider a small discount (for example, 5%) only when margin and campaign rules allow. |
| `< 30%` | Low | No incentive by default. Help the visitor discover relevant products and value; a discount may not solve the reason for low intent. |

In [6]:
def propensity_band(probability):
    if probability is None or pd.isna(probability):
        return "Unknown"
    if probability >= 0.65:
        return "High"
    if probability >= 0.30:
        return "Medium"
    return "Low"

def decision_for_band(band):
    decisions = {
        "High": "No discount by default. Make checkout easy and reinforce trust; this visitor may already be ready to convert.",
        "Medium": "Test free shipping as the first incentive. If free shipping is unavailable or unsuitable, consider a small discount (for example, 5%) only when margin and campaign rules allow.",
        "Low": "No incentive by default. Help the visitor discover relevant products and value; a discount may not solve the reason for low intent.",
        "Unknown": "Do not personalize based on propensity until the score is available and validated."
    }
    return decisions[band]

results_df["Demo propensity band"] = results_df["Conversion probability (%)"].apply(
    lambda x: propensity_band(x / 100 if pd.notna(x) else None)
)
results_df["Offer Strategy"] = results_df["Demo propensity band"].apply(decision_for_band)

display(results_df[[
    "Scenario", "Predicted class", "Class 0 probability", "Class 1 probability",
    "Conversion probability (%)", "Demo propensity band", "Offer Strategy"
]].style.format({
    "Class 0 probability": "{:.1%}",
    "Class 1 probability": "{:.1%}",
    "Conversion probability (%)": "{:.2f}%",
}, na_rep="N/A"))

,Scenario,Predicted class,Class 0 probability,Class 1 probability,Conversion probability (%),Demo propensity band,Offer Strategy
0,Quick Visit,0,97.0%,3.0%,3.00%,Low,No incentive by default. Help the visitor discover relevant products and value; a discount may not solve the reason for low intent.
1,Product Explorer,0,76.0%,24.0%,24.00%,Low,No incentive by default. Help the visitor discover relevant products and value; a discount may not solve the reason for low intent.
2,Engaged Shopper,1,48.0%,52.0%,52.00%,Medium,"Test free shipping as the first incentive. If free shipping is unavailable or unsuitable, consider a small discount (for example, 5%) only when margin and campaign rules allow."
3,Active Checkout,1,22.0%,78.0%,78.00%,High,No discount by default. Make checkout easy and reinforce trust; this visitor may already be ready to convert.


## 5. Scenario-by-scenario personalization rationale

The model score sets the illustrative propensity band; the observed behavior suggests which content might be useful. The action should remain context-aware rather than blindly applying a discount based on a score.

In [7]:
scenario_lookup = {s["scenario"]: s for s in scenarios}

for _, row in results_df.iterrows():
    scenario = scenario_lookup[row["Scenario"]]
    display(Markdown(
        f"### {row['Scenario']} - {row['Demo propensity band']} propensity\n"
        f"- **Model output:** class `{row['Predicted class']}`, class 0 probability `{row['Class 0 probability']:.1%}`, "
        f"class 1 probability `{row['Class 1 probability']:.1%}`.\n"
        f"- **Behavioral hypothesis:** {scenario['business_hypothesis']}\n"
        f"- **Band-based decision principle:** {row['Offer Strategy']}"
    ))

### Quick Visit - Low propensity
- **Model output:** class `0`, class 0 probability `97.0%`, class 1 probability `3.0%`.
- **Behavioral hypothesis:** A visitor arrives and leaves with little exploration.
- **Band-based decision principle:** No incentive by default. Help the visitor discover relevant products and value; a discount may not solve the reason for low intent.

### Product Explorer - Low propensity
- **Model output:** class `0`, class 0 probability `76.0%`, class 1 probability `24.0%`.
- **Behavioral hypothesis:** A visitor browses several pages, views multiple products and adds them to the cart without reaching checkout page
- **Band-based decision principle:** No incentive by default. Help the visitor discover relevant products and value; a discount may not solve the reason for low intent.

### Engaged Shopper - Medium propensity
- **Model output:** class `1`, class 0 probability `48.0%`, class 1 probability `52.0%`.
- **Behavioral hypothesis:** A visitor is engaged and views multiple products and reaches the checkout page, suggesting active consideration.
- **Band-based decision principle:** Test free shipping as the first incentive. If free shipping is unavailable or unsuitable, consider a small discount (for example, 5%) only when margin and campaign rules allow.

### Active Checkout - High propensity
- **Model output:** class `1`, class 0 probability `22.0%`, class 1 probability `78.0%`.
- **Behavioral hypothesis:** A visitor has added items to the cart, started checkout and clearly demostrated purchase intentions
- **Band-based decision principle:** No discount by default. Make checkout easy and reinforce trust; this visitor may already be ready to convert.